### Завдання 1. Завантаження даних
Для кожної з адміністративних одиниць України завантажити (`urllib`) тестові структуровані файли VHI-індексу. До імені файлу додати дату та час завантаження. Реалізувати механізм запобігання повторного довантаження файлів.

In [5]:
import os
import urllib.request
from datetime import datetime

directory = 'vhi_dataset'
if not os.path.exists(directory):
    os.makedirs(directory)

def fetch_vhi_data():
    for prov_id in range(1, 26):
        is_exist = any(f"prov_{prov_id}_" in filename for filename in os.listdir(directory))
        
        if not is_exist:
            url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={prov_id}&year1=1981&year2=2024&type=Mean"
            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
            file_path = os.path.join(directory, f"prov_{prov_id}_{timestamp}.csv")
            
            try:
                urllib.request.urlretrieve(url, file_path)
                print(f"Завантажено: Область {prov_id}")
            except Exception as e:
                print(f"Помилка для області {prov_id}: {e}")
        else:
            print(f"Область {prov_id} вже присутня в папці.")

fetch_vhi_data()

Область 1 вже присутня в папці.
Область 2 вже присутня в папці.
Область 3 вже присутня в папці.
Область 4 вже присутня в папці.
Область 5 вже присутня в папці.
Область 6 вже присутня в папці.
Область 7 вже присутня в папці.
Область 8 вже присутня в папці.
Область 9 вже присутня в папці.
Область 10 вже присутня в папці.
Область 11 вже присутня в папці.
Область 12 вже присутня в папці.
Область 13 вже присутня в папці.
Область 14 вже присутня в папці.
Область 15 вже присутня в папці.
Область 16 вже присутня в папці.
Область 17 вже присутня в папці.
Область 18 вже присутня в папці.
Область 19 вже присутня в папці.
Область 20 вже присутня в папці.
Область 21 вже присутня в папці.
Область 22 вже присутня в папці.
Область 23 вже присутня в папці.
Область 24 вже присутня в папці.
Область 25 вже присутня в папці.


### Завдання 2. Зчитування та очищення даних (Data Cleaning)
Зчитати завантажені текстові файли у `pandas dataframe`. Прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст. Додати стовпчики з назвою та індексом області.

In [6]:
import pandas as pd
import os

def load_and_clean_data(folder):
    dataset_list = []
    
    for file in os.listdir(folder):
        if file.endswith(".csv"):
            region_id = int(file.split('_')[1])
            
            df = pd.read_csv(os.path.join(folder, file), index_col=False, header=1,
                             names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
            
            df = df.dropna()
            
            df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '', regex=False)
            df = df[df['Year'] != 'Year']  
            df['Year'] = df['Year'].astype(int)
            df['Week'] = df['Week'].astype(int)
            
            df['Old_ID'] = region_id
            
            dataset_list.append(df)
            
    return pd.concat(dataset_list).reset_index(drop=True)

main_df = load_and_clean_data(directory)
print("Очищення пройшло успішно!")
main_df.head()

Очищення пройшло успішно!


,Year,Week,SMN,SMT,VCI,TCI,VHI,Old_ID
0,1982,1,0.059,258.24,51.11,48.78,49.95,10
1,1982,2,0.063,261.53,55.89,38.20,47.04,10
2,1982,3,0.063,263.45,57.30,32.69,44.99,10
3,1982,4,0.061,265.10,53.96,28.62,41.29,10
4,1982,5,0.058,266.42,46.87,28.57,37.72,10


### Завдання 3. Зміна індексів областей
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [7]:
# Словник: Старий ID -> (Новий ID, Назва області)
ua_index_mapping = {
    24: (1, "Вінницька"), 25: (2, "Волинська"), 5: (3, "Дніпропетровська"),
    6: (4, "Донецька"), 23: (5, "Житомирська"), 20: (6, "Закарпатська"),
    21: (7, "Запорізька"), 8: (8, "Івано-Франківська"), 9: (9, "Київська"),
    10: (10, "Кіровоградська"), 11: (11, "Луганська"), 12: (12, "Львівська"),
    13: (13, "Миколаївська"), 14: (14, "Одеська"), 15: (15, "Полтавська"),
    16: (16, "Рівненська"), 17: (17, "Сумська"), 18: (18, "Тернопільська"),
    19: (19, "Харківська"), 1: (20, "Херсонська"), 2: (21, "Хмельницька"),
    3: (22, "Черкаська"), 4: (23, "Чернівецька"), 7: (24, "Чернігівська"),
    22: (25, "Крим")
}

def apply_ua_indexing(df):
    df['Province_ID'] = df['Old_ID'].map(lambda x: ua_index_mapping.get(x, (x, "Невідомо"))[0])
    df['Province_Name'] = df['Old_ID'].map(lambda x: ua_index_mapping.get(x, (x, "Невідомо"))[1])
    
    # Видаляємо старий ID для чистоти
    return df.drop(columns=['Old_ID'])

main_df = apply_ua_indexing(main_df)
main_df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.059,258.24,51.11,48.78,49.95,10,Кіровоградська
1,1982,2,0.063,261.53,55.89,38.20,47.04,10,Кіровоградська
2,1982,3,0.063,263.45,57.30,32.69,44.99,10,Кіровоградська
3,1982,4,0.061,265.10,53.96,28.62,41.29,10,Кіровоградська
4,1982,5,0.058,266.42,46.87,28.57,37.72,10,Кіровоградська


### Завдання 4. Вибірка ряду VHI
Реалізувати процедуру для формування вибірки ряду VHI для області за вказаний рік.

In [ ]:
def get_vhi_yearly(df, province_id, year):
    filtered = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    return filtered[['Week', 'VHI']]

print("Вибірка VHI для Вінницької області (ID: 1) за 2019 рік:")
get_vhi_yearly(main_df, 1, 2019).head(10)

Вибірка VHI для Вінницької області (ID: 1) за 2019 рік:


,Week,VHI
35464,1,50.77
35465,2,51.84
35466,3,53.97
35467,4,52.17
35468,5,48.17
35469,6,46.06
35470,7,44.33
35471,8,43.87
35472,9,45.33
35473,10,46.33


### Завдання 5. Пошук екстремумів
Реалізувати процедуру пошуку екстремумів (min та max) для вказаних областей та років.

In [9]:
def find_vhi_extremes(df, province_id, year):
    data_subset = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]['VHI']
    
    if data_subset.empty:
        return "Дані за цей період відсутні."
        
    return {
        "Область ID": province_id,
        "Рік": year,
        "Мінімум VHI": data_subset.min(),
        "Максимум VHI": data_subset.max()
    }

print("Екстремуми для Київської області (ID: 9) за 2021 рік:")
find_vhi_extremes(main_df, 9, 2021)

Екстремуми для Київської області (ID: 9) за 2021 рік:


{'Область ID': 9,
 'Рік': 2021,
 'Мінімум VHI': np.float64(26.91),
 'Максимум VHI': np.float64(76.44)}